# Task 1 — Claim Detection training on Colab

**Before running**
1. On your laptop, in the project root: `bash notebooks/make_task1_bundle.sh` → creates `outputs/task1_colab.zip` (~1.4 MB).
2. Upload `task1_colab.zip` to Google Drive, folder **`MyDrive/nlp-final-project/`** (replace the old one if it exists).
3. In Colab: **Runtime → Change runtime type → T4 GPU**.
4. **Runtime → Run all**, or run cells one by one to skip the optional deberta run. Each run takes roughly 10–15 minutes.

**Re-running is safe.** The setup cell only overwrites code and data. Models already trained in this session are kept,
and each run is saved to Drive as its own zip.

**After it finishes**
- From `MyDrive/nlp-final-project/`, download each `task1_<model>_<mapping>.zip` (~0.5 GB each).
- Unzip each one in the project root on your laptop, so you get `models/task1_roberta-base_label/model/...`.
- `ClaimDetector()` in `src/task1_claim_detection/infer.py` loads that folder by default.

In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/nlp-final-project'
assert os.path.exists(f'{DRIVE_DIR}/task1_colab.zip'), f'Upload task1_colab.zip to {DRIVE_DIR} first'

In [ ]:
# Overwrite code + data only; keep any models/ already trained in this session
!mkdir -p /content/project
!unzip -oq "{DRIVE_DIR}/task1_colab.zip" -d /content/project
%cd /content/project
!ls -R data src | head -30
!ls models 2>/dev/null || echo "(no models trained yet in this session)"

In [ ]:
!pip install -q "transformers>=4.46" accelerate sentencepiece
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())

## 1. Main run — factual (CFS + UFS) vs non-factual  (plan 1.2)

In [ ]:
!python -m src.task1_claim_detection.train --model roberta-base --label-col label --batch-size 32 --grad-accum 1

## 2. Comparison run — original check-worthiness mapping, CFS only  (plan 1.3)

In [ ]:
!python -m src.task1_claim_detection.train --model roberta-base --label-col label_checkworthy --batch-size 32 --grad-accum 1

## 3. (Optional) Second backbone from the plan — deberta-v3-small
Skip this cell if you are short on time or GPU quota.

In [ ]:
!python -m src.task1_claim_detection.train --model microsoft/deberta-v3-small --label-col label --batch-size 32 --grad-accum 1

## 4. Summary of all runs

In [ ]:
import glob, json
rows = []
for path in sorted(glob.glob('models/*/metrics.json')):
    m = json.load(open(path))
    if 'test@tuned' not in m:      # skip the tf-idf baseline file if present
        continue
    t = m['test@tuned']
    rows.append((m['run'], m['tuned_threshold'], t['macro_f1'], t['precision'], t['recall'], t['roc_auc']))
print(f"{'run':<45}{'thr':>6}{'macroF1':>9}{'prec':>8}{'recall':>8}{'rocAUC':>8}")
for r in rows:
    print(f"{r[0]:<45}{r[1]:>6.2f}{r[2]:>9.4f}{r[3]:>8.4f}{r[4]:>8.4f}{r[5]:>8.4f}")

## 5. Save trained models to Drive

In [ ]:
# One zip per run, so re-running a single model never overwrites the others
import glob, os, subprocess
for run_dir in sorted(glob.glob('models/task1_*')):
    if not os.path.exists(f'{run_dir}/model/model.safetensors'):
        continue
    out = f'{DRIVE_DIR}/{os.path.basename(run_dir)}.zip'
    if os.path.exists(out):
        os.remove(out)
    subprocess.run(['zip', '-qr', out, run_dir, '-x', f'{run_dir}/trainer/*'], check=True)
    print(f'{out}  ({os.path.getsize(out) / 1e6:.0f} MB)')